In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("gk_qna_dataset.csv")

In [4]:
df

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo
...,...,...
166,"Can you tell me, ""Which bird cannot fly""?",Ostrich
167,"I was wondering, ""What is 'H2O' commonly called""?",Water
168,"I was wondering, ""Which planet is known as the...",Mars
169,"Can you tell me, ""What is the main religion in...",Islam


In [5]:
df["question"]

0                       What is the capital of "France"?
1                      What is the capital of "Germany"?
2                        What is the capital of "Italy"?
3                        What is the capital of "Spain"?
4                        What is the capital of "Japan"?
                             ...                        
166            Can you tell me, "Which bird cannot fly"?
167    I was wondering, "What is 'H2O' commonly called"?
168    I was wondering, "Which planet is known as the...
169    Can you tell me, "What is the main religion in...
170    Quick question: "Which animal is known for loy...
Name: question, Length: 171, dtype: object

In [7]:
df["question"][0]

'What is the capital of "France"?'

tokenize

In [9]:
import re

In [10]:
def tokenize(text): 
    text = text.lower() 

    text = re.sub(r"[\"']","",text)  
                  
    text = re.sub(r"[^a-z0-9\s]"," ",text) 

    text = re.sub(r"\s+"," ", text).strip() 

    tokens = text.split() 

    return tokens

In [11]:
print(tokenize(df['question'][0]))

['what', 'is', 'the', 'capital', 'of', 'france']


In [13]:
vocab = {'<UNK>':0}

In [14]:
def build_vocab(row): 
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_tokens = tokenized_question + tokenized_answer 

    for token in merged_tokens: 
        if token not in vocab: 
            vocab[token] = len(vocab)

In [15]:
df.apply(build_vocab,axis = 1)

0      None
1      None
2      None
3      None
4      None
       ... 
166    None
167    None
168    None
169    None
170    None
Length: 171, dtype: object

In [16]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'italy': 10,
 'rome': 11,
 'spain': 12,
 'madrid': 13,
 'japan': 14,
 'tokyo': 15,
 'canada': 16,
 'ottawa': 17,
 'brazil': 18,
 'brasilia': 19,
 'australia': 20,
 'canberra': 21,
 'india': 22,
 'new': 23,
 'delhi': 24,
 'china': 25,
 'beijing': 26,
 'russia': 27,
 'moscow': 28,
 'united': 29,
 'states': 30,
 'washington': 31,
 'dc': 32,
 'mexico': 33,
 'city': 34,
 'egypt': 35,
 'cairo': 36,
 'turkey': 37,
 'ankara': 38,
 'argentina': 39,
 'buenos': 40,
 'aires': 41,
 'south': 42,
 'korea': 43,
 'seoul': 44,
 'indonesia': 45,
 'jakarta': 46,
 'pakistan': 47,
 'islamabad': 48,
 'bangladesh': 49,
 'dhaka': 50,
 'nepal': 51,
 'kathmandu': 52,
 'sri': 53,
 'lanka': 54,
 'colombo': 55,
 'thailand': 56,
 'bangkok': 57,
 'malaysia': 58,
 'kuala': 59,
 'lumpur': 60,
 'vietnam': 61,
 'hanoi': 62,
 'uae': 63,
 'abu': 64,
 'dhabi': 65,
 'iran': 66,
 'tehran': 67,
 'iraq

In [17]:
len(vocab)

243

In [18]:
vocab['what']

1

In [20]:
def text_to_indices(text,vocab):
  indexed_text = []

  for token in tokenize(text):
    
    if token in vocab:
      
      indexed_text.append(vocab[token])

    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text


In [21]:
text_to_indices("the is Phitron",vocab)

[3, 2, 0]

Dataset and Dataloader

In [22]:
import torch
from torch.utils.data import Dataset,DataLoader

In [23]:
df.shape[0]

171

In [24]:
index = 0

text_to_indices(df.iloc[index]['question'],vocab)

[1, 2, 3, 4, 5, 6]

In [25]:
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [26]:
dataset = QADataset(df,vocab)

In [27]:
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [28]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [29]:
for question,answer in dataloader:
  print(question,answer[0])

tensor([[ 79,  84,   3, 161, 162,  95, 232, 233]]) tensor([163, 164])
tensor([[230, 231,  79, 170, 171]]) tensor([172, 173])
tensor([[240, 232, 241, 242,  93, 215, 216, 218]]) tensor([219])
tensor([[240, 232, 241, 242,   1, 131, 132, 133, 121]]) tensor([134, 130, 135])
tensor([[237,   1, 238, 239,   1,   2,  89,  90,  91]]) tensor([92])
tensor([[ 93, 197,   2, 200]]) tensor([201])
tensor([[  1,   2, 136]]) tensor([137, 138])
tensor([[  1, 131, 132, 133, 121]]) tensor([134, 130, 135])
tensor([[230, 231,  93, 179,   2, 101, 102,   3, 180,   5, 181, 182]]) tensor([14])
tensor([[240, 232, 241, 242,   1,   2, 124, 120, 121]]) tensor([125, 126])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[234, 235, 236,   1,   2, 127, 120, 121]]) tensor([128, 129, 130])
tensor([[ 95, 232, 233,  79, 170, 174, 175]]) tensor([176, 177, 178])
tensor([[234, 235, 236,   1, 131, 132, 133, 121]]) tensor([134, 130, 135])
tensor([[240, 232, 241, 242,   1,   2,   3, 111, 112]]) tensor([113, 112])
tensor([[

In [30]:
# squeeze

import torch

x = torch.tensor( [[[1,2,3]]] )



y = x.squeeze(0)

print(y.shape)

torch.Size([1, 3])


RNN Architecture 
Implementation

In [31]:
import torch.nn as nn

class simpleRNN(nn.Module):
  def __init__(self,vocab_size,embedding_dim=50,hidden_size=64):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size,embedding_dim)
    self.rnn = nn.RNN(embedding_dim,hidden_size,batch_first=True)
    self.fc = nn.Linear(hidden_size,vocab_size)


  def forward(self,question):
     embedded = self.embedding(question) 
     _,final = self.rnn(embedded)

     return self.fc(final.squeeze(0))  


In [32]:
model = simpleRNN(vocab_size = len(vocab))

In [33]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
epochs = 50

In [34]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        output = model(question)
        
        target = answer[0][0].unsqueeze(0)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs}  |  Loss: {total_loss:.4f}")


Epoch  10/50  |  Loss: 588.3247
Epoch  20/50  |  Loss: 375.8210
Epoch  30/50  |  Loss: 251.9733
Epoch  40/50  |  Loss: 178.3295
Epoch  50/50  |  Loss: 127.1385


In [35]:
vocab.items()

dict_items([('<UNK>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('of', 5), ('france', 6), ('paris', 7), ('germany', 8), ('berlin', 9), ('italy', 10), ('rome', 11), ('spain', 12), ('madrid', 13), ('japan', 14), ('tokyo', 15), ('canada', 16), ('ottawa', 17), ('brazil', 18), ('brasilia', 19), ('australia', 20), ('canberra', 21), ('india', 22), ('new', 23), ('delhi', 24), ('china', 25), ('beijing', 26), ('russia', 27), ('moscow', 28), ('united', 29), ('states', 30), ('washington', 31), ('dc', 32), ('mexico', 33), ('city', 34), ('egypt', 35), ('cairo', 36), ('turkey', 37), ('ankara', 38), ('argentina', 39), ('buenos', 40), ('aires', 41), ('south', 42), ('korea', 43), ('seoul', 44), ('indonesia', 45), ('jakarta', 46), ('pakistan', 47), ('islamabad', 48), ('bangladesh', 49), ('dhaka', 50), ('nepal', 51), ('kathmandu', 52), ('sri', 53), ('lanka', 54), ('colombo', 55), ('thailand', 56), ('bangkok', 57), ('malaysia', 58), ('kuala', 59), ('lumpur', 60), ('vietnam', 61), ('hanoi', 62

In [36]:
idx_to_word = { idx : word for word,idx in vocab.items()  }
idx_to_word

{0: '<UNK>',
 1: 'what',
 2: 'is',
 3: 'the',
 4: 'capital',
 5: 'of',
 6: 'france',
 7: 'paris',
 8: 'germany',
 9: 'berlin',
 10: 'italy',
 11: 'rome',
 12: 'spain',
 13: 'madrid',
 14: 'japan',
 15: 'tokyo',
 16: 'canada',
 17: 'ottawa',
 18: 'brazil',
 19: 'brasilia',
 20: 'australia',
 21: 'canberra',
 22: 'india',
 23: 'new',
 24: 'delhi',
 25: 'china',
 26: 'beijing',
 27: 'russia',
 28: 'moscow',
 29: 'united',
 30: 'states',
 31: 'washington',
 32: 'dc',
 33: 'mexico',
 34: 'city',
 35: 'egypt',
 36: 'cairo',
 37: 'turkey',
 38: 'ankara',
 39: 'argentina',
 40: 'buenos',
 41: 'aires',
 42: 'south',
 43: 'korea',
 44: 'seoul',
 45: 'indonesia',
 46: 'jakarta',
 47: 'pakistan',
 48: 'islamabad',
 49: 'bangladesh',
 50: 'dhaka',
 51: 'nepal',
 52: 'kathmandu',
 53: 'sri',
 54: 'lanka',
 55: 'colombo',
 56: 'thailand',
 57: 'bangkok',
 58: 'malaysia',
 59: 'kuala',
 60: 'lumpur',
 61: 'vietnam',
 62: 'hanoi',
 63: 'uae',
 64: 'abu',
 65: 'dhabi',
 66: 'iran',
 67: 'tehran',
 68: '

In [37]:
def predict(question_text,model,vocab,idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indices(question_text,vocab)

    if not indices:
      return "<UNK>"

    x = torch.tensor(indices).unsqueeze(0)  #batch size, sequence legth

    pred = torch.argmax(model(x),dim=1).item()


  return idx_to_word.get(pred,"<UNK>")

In [38]:
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",

]
for q in tests:
    print(f"{q:45} → {predict(q, model, vocab, idx_to_word)}")

What is the capital of France?                → paris
What is H2O commonly called?                  → water
Which planet is known as the red planet?      → mars
Which bird cannot fly?                        → ostrich
What is the capital of Japan ?                → tokyo
